<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h1>Essential DataFrame Methods</h1>
DAT-7470 - Business Analysis with Structured Data<br>
Script 02 - Completed Notebook<br><br>
Based on the course script by Chase Kusterer, Hult International Business School<br>
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h2>Part I: Practice - Importing Excel Data into Python</h2>
Each new session starts with an empty environment, so the data has to be imported again.<br><br>
<strong>Google Colab:</strong> upload <code>diamonds.xlsx</code> with the folder icon on the left (Files &rarr; Upload), or run the cell below and pick the file when the upload button appears.<br>
<strong>Local Jupyter:</strong> keep the file next to this notebook or in a <code>datasets</code> folder.

<h4>a) Import pandas and the diamonds dataset.</h4>

In [1]:
# importing packages as pd
import pandas as pd
import os
import warnings

# hiding version warnings so the output stays clean
warnings.filterwarnings("ignore")


# storing the path to the dataset
# (works in Google Colab and in local Jupyter)
file = None
for path in ["./diamonds.xlsx", "./datasets/diamonds.xlsx", "/content/diamonds.xlsx"]:
    if os.path.exists(path):
        file = path
        break

if file is None:                       # not found -> open Colab's upload button
    from google.colab import files
    uploaded = files.upload()          # choose diamonds.xlsx from your computer
    file = list(uploaded.keys())[0]


# reading the file into Python through pandas
diamonds = pd.read_excel(file)


# checking results (first TEN rows of the dataset)
diamonds.head(n = 10)

,Obs,carat,color,clarity,cut,channel,store,price
0,1,0.32,3,7,0,0,7,1375
1,2,0.35,5,7,1,0,7,1680
2,3,0.40,4,7,1,0,7,2245
3,4,0.59,5,7,0,0,3,2495
4,5,0.72,7,8,0,0,8,2699
5,6,0.72,6,8,0,0,8,2699
6,7,0.51,2,8,0,0,3,2895
7,8,1.00,8,9,0,0,7,2900
8,9,1.00,8,9,0,0,8,2999
9,10,0.47,2,7,0,0,7,3080


<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h2>Part II: Auditing Data Quality - Fixing Feature Types</h2>
After importing, we check data quality. Three methods are key for this:

* `.info()` – column names, non-null counts and data types
* `.describe()` – descriptive statistics
* `.value_counts()` – how often each value appears

We start with `.info()`.

In [2]:
# analyzing info related to each feature
diamonds.info(verbose = True)

<class 'pandas.DataFrame'>
RangeIndex: 409 entries, 0 to 408
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Obs      409 non-null    int64  
 1   carat    409 non-null    float64
 2   color    409 non-null    int64  
 3   clarity  409 non-null    int64  
 4   cut      409 non-null    int64  
 5   channel  409 non-null    int64  
 6   store    409 non-null    int64  
 7   price    409 non-null    int64  
dtypes: float64(1), int64(7)
memory usage: 25.7 KB


<strong>How to read <code>.info()</code>:</strong>

* **RangeIndex** – number of rows (409, index 0 to 408).
* **Column / Non-Null Count** – every column has 409 non-null values, so there are **no missing values**.
* **Dtype** – `float64` (decimals) for carat, `int64` (whole numbers) for the rest.
* **memory usage** – size of the DataFrame in memory (beyond the scope of this course).

<h4>a) From the output above, which features seem to have improper data types?</h4>

In [3]:
# features with improper data types
#    Obs
#    channel
#    store
#    cut (potentially)


# Rationale for above
"""
Obs, channel and store are stored as integers, but they are labels, not
measurements. Their numbers only identify a row, a sales channel
(0 = Mall, 1 = Independent, 2 = Internet) or a store. Their order has no
meaning: store 10 is not "bigger" than store 2, and the average of the channel
column means nothing. They should be treated as categories (strings).

Obs is just a row ID, so it should not be used in the analysis at all.

Cut can also be questioned. The documentation says it answers a yes/no
question (Ideal or Not Ideal). A 0/1 code works for this, but in reality cut
quality is a scale with shades of gray, so a single 0/1 number hides
information.
"""

'\nObs, channel and store are stored as integers, but they are labels, not\nmeasurements. Their numbers only identify a row, a sales channel\n(0 = Mall, 1 = Independent, 2 = Internet) or a store. Their order has no\nmeaning: store 10 is not "bigger" than store 2, and the average of the channel\ncolumn means nothing. They should be treated as categories (strings).\n\nObs is just a row ID, so it should not be used in the analysis at all.\n\nCut can also be questioned. The documentation says it answers a yes/no\nquestion (Ideal or Not Ideal). A 0/1 code works for this, but in reality cut\nquality is a scale with shades of gray, so a single 0/1 number hides\ninformation.\n'

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h3>Fixing Data Types</h3>
We change data types with a dictionary. We only list the features we want to change:

~~~
{"feature" : data type,     # feature 1
 "feature" : data type,     # feature 2
 ...}                       # and so on
~~~

We usually apply this in the same step as importing the data.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>b) Convert data types as indicated by the table below.</h4>
<em>Obs</em> is left out because its order is arbitrary and it is not used in the analysis.

| Feature | Original Type | New Type | Rationale for Change |
|---|---|---|---|
| channel | int | str | feature is categorical (pre-processing step) |
| store | int | str | feature is categorical (pre-processing step) |

The help file shows that the **`usecols`** argument picks which columns to import.

In [4]:
# help file for pd.read_excel
help(pd.read_excel)

Help on function read_excel in module pandas:

read_excel(
    io,
    sheet_name: 'str | int | list[IntStrT] | None' = 0,
    *,
    header: 'int | Sequence[int] | None' = 0,
    names: 'SequenceNotStr[Hashable] | range | None' = None,
    index_col: 'int | str | Sequence[int] | None' = None,
    usecols: 'int | str | Sequence[int] | Sequence[str] | Callable[[HashableT], bool] | None' = None,
    dtype: 'DtypeArg | None' = None,
    engine: "Literal['xlrd', 'openpyxl', 'odf', 'pyxlsb', 'calamine'] | None" = None,
    converters: 'dict[str, Callable] | dict[int, Callable] | None' = None,
    true_values: 'Iterable[Hashable] | None' = None,
    false_values: 'Iterable[Hashable] | None' = None,
    skiprows: 'Sequence[int] | int | Callable[[int], object] | None' = None,
    nrows: 'int | None' = None,
    na_values=None,
    keep_default_na: 'bool' = True,
    na_filter: 'bool' = True,
    verbose: 'bool' = False,
    parse_dates: 'list | dict | bool' = False,
    date_format: 'dict[Hash

In [5]:
# converting data types with a dictionary
data_types = {"channel" : str,
              "store"   : str}


# instantiating the dataset as an object
diamonds = pd.read_excel(io         = file      ,
                         dtype      = data_types,   # applying new data types
                         usecols    = range(1, 8) ) # columns 1 to 7 (skips Obs, column 0)


# checking results
diamonds.info(verbose = True)

<class 'pandas.DataFrame'>
RangeIndex: 409 entries, 0 to 408
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    409 non-null    float64
 1   color    409 non-null    int64  
 2   clarity  409 non-null    int64  
 3   cut      409 non-null    int64  
 4   channel  409 non-null    str    
 5   store    409 non-null    str    
 6   price    409 non-null    int64  
dtypes: float64(1), int64(4), str(2)
memory usage: 22.5 KB


<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<em>Obs</em> is gone, and <em>channel</em> and <em>store</em> are now text.
In Colab (pandas 2) they show as type <strong>object</strong>. This is normal: when pandas stores text it uses the general <em>object</em> type, because a string can represent almost anything. Newer pandas versions (3.0+) show <strong>str</strong> instead. Both mean the same thing.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h2>Part III: Analyzing Anomalies with Descriptive Statistics</h2>
Questions for `.describe()`:

* Does the data match its documentation?
* Are there extreme values that skew any feature?

Assumptions from the <em>"Best" Diamond on a Budget</em> case study to check:

* Is \$5,795 a reasonable amount for a normal citizen of Chicago (close to the center of price)?
* Can a 5 on the color and clarity scales be seen as "normal"?

<h3>Basic Use of .describe()</h3>

In [6]:
# crude example of .describe()
diamonds.describe()

,carat,color,clarity,cut,price
count,409.000000,409.000000,409.000000,409.000000,409.000000
mean,1.055968,4.227384,5.056235,0.376528,6488.229829
std,0.418013,1.810815,1.573141,0.485108,4414.425161
min,0.320000,1.000000,1.000000,0.000000,878.000000
25%,0.720000,3.000000,4.000000,0.000000,3490.000000
50%,1.020000,4.000000,5.000000,0.000000,5550.000000
75%,1.210000,6.000000,6.000000,1.000000,8089.000000
max,2.480000,9.000000,9.000000,1.000000,27575.000000


<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
Too many decimal places are <strong><font style="color:red">another form of data dumping.</font></strong>
Because the type changes worked, <em>channel</em> and <em>store</em> are no longer in this table.

<h3>Properly Applying .describe()</h3>
<h4>a) Output descriptive statistics for the numeric features, rounded to two decimal places.</h4>

In [7]:
# descriptive statistics on numeric columns
diamonds.describe(include = 'number').round(decimals = 2)

,carat,color,clarity,cut,price
count,409.00,409.00,409.00,409.00,409.00
mean,1.06,4.23,5.06,0.38,6488.23
std,0.42,1.81,1.57,0.49,4414.43
min,0.32,1.00,1.00,0.00,878.00
25%,0.72,3.00,4.00,0.00,3490.00
50%,1.02,4.00,5.00,0.00,5550.00
75%,1.21,6.00,6.00,1.00,8089.00
max,2.48,9.00,9.00,1.00,27575.00


<strong>What the table tells us:</strong>

* **Matches the documentation:** color and clarity run from 1 to 9, cut is only 0 or 1, and no column has missing values.
* **Price is skewed right.** The mean (\$6,488) is well above the median (\$5,550), and the maximum is \$27,575, about 4 standard deviations above the mean. A few very expensive diamonds pull the average up, so the **median is the better "center"**.
* **Carat:** values from 0.32 to 2.48, with mean 1.06 and median 1.02. The 2.48 carat stone is an extreme value, but the distribution is much less skewed than price.
* **Is \$5,795 reasonable?** Yes. It sits between the median (\$5,550) and the mean (\$6,488), close to the center of the distribution. About 53% of the diamonds cost \$5,795 or less.
* **Is 5 "normal" for color and clarity?** For clarity, yes: the median is 5 and the mean is 5.06. For color, 5 is slightly worse than typical, since the median is 4 and the mean is 4.23. A color of 4 or 5 is a fair "normal" range.
* **Cut:** the mean of 0.38 means only 38% of the diamonds are Ideal cut.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>b) Output frequency information for the non-numeric features.</h4>

In [8]:
# descriptive statistics on non-numeric columns
diamonds.describe(include = 'object')

,channel,store
count,409,409
unique,3,11
top,2,10
freq,318,211


<strong>How to read this table:</strong>

* **count** – number of non-missing values (409)
* **unique** – number of different categories (3 channels, 11 stores)
* **top** – the most common category (channel 2 = Internet, store 10)
* **freq** – how many times the top category appears (318 and 211)

This is useful but limited, so we also use `.value_counts()`, which works on numeric and non-numeric data.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>c) Apply .value_counts() to channel.</h4>
Does each channel have equal representation? What does this imply?

In [9]:
# applying value_counts to channel
diamonds['channel'].value_counts(normalize = False,
                                 sort      = False,
                                 ascending = False)

channel
0     43
1     48
2    318
Name: count, dtype: int64

In [10]:
# same result as percentages, to make the comparison easier
diamonds['channel'].value_counts(normalize = True).round(decimals = 3)

channel
2    0.778
1    0.117
0    0.105
Name: proportion, dtype: float64

<strong>Answer:</strong> No. The channels are **not equally represented**. 318 diamonds (about 78%) come from the Internet (channel 2), 48 (12%) from Independent jewelers (channel 1), and only 43 (10.5%) from Malls (channel 0).

<strong>Implications:</strong>
* Any overall statistic (average price, typical color, and so on) mostly describes **online** diamonds.
* Conclusions about malls and independent jewelers rest on small samples, so they are less reliable.
* To compare channels fairly, we should look at each channel separately (for example with `groupby`) rather than at the total.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>d) Apply .value_counts() to store.</h4>
Set the sort argument to True. Do any of the stores have less than 30 observations?

In [11]:
# applying value_counts to store
diamonds['store'].value_counts(normalize = False, # percentages
                               sort      = True , # sort by frequencies
                               ascending = False) # smallest value on top?

store
10    211
11    107
3      15
7      14
6      13
1      11
9       9
2       8
8       7
4       7
5       7
Name: count, dtype: int64

In [12]:
# stores with fewer than 30 observations
store_counts = diamonds['store'].value_counts()
store_counts[ store_counts < 30 ]

store
3    15
7    14
6    13
1    11
9     9
2     8
8     7
4     7
5     7
Name: count, dtype: int64

<strong>Answer:</strong> Yes. **9 of the 11 stores** have fewer than 30 observations (between 7 and 15 each). Only store 10 (211) and store 11 (107) have more than 30. Those two stores are the online sellers and together hold about 78% of the data.

30 is a common rule of thumb for the minimum sample size needed to make reliable statistical statements. So we should **not draw strong conclusions about any single small store**. If store-level analysis is needed, the small stores could be grouped together (for example by channel).

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>e) Apply .value_counts() to color.</h4>
Set normalize to True and round to three decimal places. Is this feature (relatively) normally distributed?

In [13]:
# applying value_counts to color
diamonds['color'].value_counts(normalize = True ,
                               sort      = False,
                               ascending = False).round(decimals = 3)\
                                                 .sort_index(axis = 0,
                                                             ascending = True)

color
1    0.073
2    0.142
3    0.142
4    0.176
5    0.176
6    0.193
7    0.083
8    0.012
9    0.002
Name: proportion, dtype: float64

<strong>Answer:</strong> **Roughly, but not perfectly.** Most diamonds sit in the middle of the scale: grades 2 to 6 hold about 83% of the data, and the most common grades are 6 (19.3%), 4 and 5 (17.6% each). The lowest grades (8 and 9) are very rare (1.4% combined).

The distribution is **not symmetric**. It has a long tail on the low-quality side (7, 8, 9) and a shorter, fatter side on the high-quality end (1 is 7.3%). Most of the data is on the better-color side of the scale, so it is slightly **left-skewed** around a center of about 4 to 5. This supports using a color of about 4 to 5 as "normal".

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h4>f) Apply .value_counts() to clarity.</h4>
Set normalize to True and round to two decimal places. Is this feature (relatively) normally distributed?

In [14]:
# applying value_counts to clarity
diamonds['clarity'].value_counts(normalize = True ,
                                 sort      = False,
                                 ascending = False).round(decimals = 2)\
                                                   .sort_index(axis = 0,
                                                               ascending = True)

clarity
1    0.03
2    0.03
3    0.10
4    0.19
5    0.26
6    0.20
7    0.16
8    0.03
9    0.00
Name: proportion, dtype: float64

<strong>Answer:</strong> **Yes, relatively normal.** Clarity has one clear peak at 5 (26%), with the values falling off on both sides: 4 (19%) and 6 (20%), then 3 (10%) and 7 (16%), then very few at the extremes (1, 2, 8 and 9 are each 3% or less). The mean (5.06) and median (5) are almost equal, which also points to a fairly symmetric, bell-shaped distribution.

This confirms the assumption from the case study: **a clarity of 5 is a "normal" diamond.**

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<h3>Summary of the Data Audit</h3>

| Check | Finding |
|---|---|
| Missing values | None (409 of 409 for every column) |
| Data types | Obs, channel and store are labels, not numbers. Obs was dropped; channel and store were converted to text |
| Price | Right-skewed with extreme values up to \$27,575, so the median (\$5,550) is the best center |
| \$5,795 budget | Reasonable: close to the center, and 53% of diamonds are affordable |
| Channel | Unbalanced: 78% Internet, so results mainly describe online diamonds |
| Store | 9 of 11 stores have fewer than 30 observations, so store-level results are unreliable |
| Color | Roughly normal around 4 to 5, slightly left-skewed |
| Clarity | Relatively normal, centered on 5 |
